# Image-to-Image Translation using Pix2Pix GAN
**Sem 7 Deep Learning Mini Project**

This notebook trains a compact Pix2Pix conditional GAN on paired building-facade images. The input is a facade label/map image and the target is the corresponding real facade photo.

**How to run**
1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** if available.
3. Run the cells from top to bottom.
4. For a quick first demo, set `EPOCHS = 5`; image quality may be limited. More training usually improves results.

**Important:** This is a real training notebook, not a pretrained image-generation API. Training time and visual quality depend on the runtime and number of epochs.


## 1. Imports and settings

In [ ]:
import os, time, zipfile, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = 256
BATCH_SIZE = 1
EPOCHS = 5  # Increase later if time and GPU allow


## 2. Download and inspect the paired Facades dataset

In [ ]:
# Public Pix2Pix facades dataset used in the original pix2pix project.
DATA_URL = "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz"
ARCHIVE_PATH = "/content/facades.tar.gz"
DATA_DIR = "/content/facades"

if not os.path.exists(DATA_DIR):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    import tarfile
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall("/content")
    print("Dataset extracted.")

train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")
print("Train images:", len(list(Path(train_dir).glob("*"))))
print("Validation images:", len(list(Path(val_dir).glob("*"))))


## 3. Load paired images and preprocess

In [ ]:
def load_pair(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    width = tf.shape(image)[1]
    # Facades samples are concatenated side-by-side: target | input.
    half = width // 2
    target = image[:, :half, :]
    source = image[:, half:, :]
    source = tf.image.resize(source, [IMG_SIZE, IMG_SIZE])
    target = tf.image.resize(target, [IMG_SIZE, IMG_SIZE])
    source = tf.cast(source, tf.float32) / 127.5 - 1.0
    target = tf.cast(target, tf.float32) / 127.5 - 1.0
    return source, target

def make_dataset(folder, training=True):
    paths = sorted([str(p) for p in Path(folder).glob("*")])
    ds = tf.data.Dataset.from_tensor_slices(paths)
    if training:
        ds = ds.shuffle(min(len(paths), 400))
    ds = ds.map(load_pair, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_dir, training=True)
val_ds = make_dataset(val_dir, training=False)

example_source, example_target = next(iter(train_ds))
plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.title("Input image"); plt.imshow((example_source[0]+1)/2); plt.axis("off")
plt.subplot(1,2,2); plt.title("Target facade photo"); plt.imshow((example_target[0]+1)/2); plt.axis("off")
plt.show()


## 4. Build the U-Net Generator

In [ ]:
def downsample(filters, size, apply_batchnorm=True):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding="same",
                             kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

def upsample(filters, size, apply_dropout=False):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2DTranspose(filters, size, strides=2, padding="same",
                                      kernel_initializer=initializer, use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def make_generator():
    inputs = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3])
    downs = [
        downsample(64, 4, False), downsample(128, 4), downsample(256, 4),
        downsample(512, 4), downsample(512, 4), downsample(512, 4),
        downsample(512, 4), downsample(512, 4)
    ]
    ups = [
        upsample(512, 4, True), upsample(512, 4, True), upsample(512, 4, True),
        upsample(512, 4), upsample(256, 4), upsample(128, 4), upsample(64, 4)
    ]
    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4, strides=2, padding="same",
                                  kernel_initializer=initializer, activation="tanh")
    x = inputs
    skips = []
    for down in downs:
        x = down(x)
        skips.append(x)
    skips = reversed(skips[:-1])
    for up, skip in zip(ups, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])
    outputs = last(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="Pix2Pix_Generator")

generator = make_generator()
generator.summary()


## 5. Build the PatchGAN Discriminator

In [ ]:
def make_discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)
    inp = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3], name="input_image")
    target = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3], name="target_image")
    x = layers.Concatenate()([inp, target])
    x = downsample(64, 4, False)(x)
    x = downsample(128, 4)(x)
    x = downsample(256, 4)(x)
    x = layers.ZeroPadding2D()(x)
    x = layers.Conv2D(512, 4, strides=1, kernel_initializer=initializer, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.ZeroPadding2D()(x)
    output = layers.Conv2D(1, 4, strides=1, kernel_initializer=initializer)(x)
    return tf.keras.Model(inputs=[inp, target], outputs=output, name="PatchGAN_Discriminator")

discriminator = make_discriminator()
discriminator.summary()


## 6. Define losses and optimizers

In [ ]:
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
LAMBDA = 100

def generator_loss(disc_generated_output, gen_output, target):
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    return gan_loss + LAMBDA * l1_loss, gan_loss, l1_loss

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss


## 7. Train the GAN

In [ ]:
@tf.function
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)
        total_gen_loss, gan_loss, l1_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    gen_gradients = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gen_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_gradients, discriminator.trainable_variables))
    return total_gen_loss, disc_loss, gan_loss, l1_loss

def generate_and_show(model, test_input, target, title):
    prediction = model(test_input, training=False)
    plt.figure(figsize=(12,4))
    images = [test_input[0], target[0], prediction[0]]
    labels = ["Input", "Ground truth", "Generated output"]
    for i, (img, label) in enumerate(zip(images, labels)):
        plt.subplot(1,3,i+1)
        plt.title(label)
        plt.imshow(np.clip((img.numpy()+1)/2, 0, 1))
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

sample_input, sample_target = next(iter(val_ds))
history = {"generator": [], "discriminator": []}
start = time.time()

for epoch in range(1, EPOCHS + 1):
    gen_losses, disc_losses = [], []
    for input_image, target in train_ds:
        g_loss, d_loss, _, _ = train_step(input_image, target)
        gen_losses.append(float(g_loss))
        disc_losses.append(float(d_loss))
    history["generator"].append(float(np.mean(gen_losses)))
    history["discriminator"].append(float(np.mean(disc_losses)))
    print(f"Epoch {epoch}/{EPOCHS} | Generator loss: {history['generator'][-1]:.3f} | Discriminator loss: {history['discriminator'][-1]:.3f}")
    generate_and_show(generator, sample_input, sample_target, f"Epoch {epoch}")

print(f"Training finished in {(time.time()-start)/60:.1f} minutes.")


## 8. Plot training losses

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1, EPOCHS+1), history["generator"], marker="o", label="Generator loss")
plt.plot(range(1, EPOCHS+1), history["discriminator"], marker="o", label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pix2Pix Training Loss")
plt.grid(True)
plt.legend()
plt.show()


## 9. Try another input image

In [ ]:
# Show a different validation input and the model's generated translation.
for test_input, test_target in val_ds.skip(1).take(1):
    generate_and_show(generator, test_input, test_target, "Validation example")


## 10. Save the trained model

In [ ]:
generator.save("pix2pix_generator.keras")
discriminator.save("pix2pix_discriminator.keras")
print("Saved pix2pix_generator.keras and pix2pix_discriminator.keras")


## Project explanation for viva

- **GAN:** A Generative Adversarial Network has a Generator and a Discriminator.
- **Pix2Pix:** A conditional GAN for paired image-to-image translation.
- **Generator:** Uses a U-Net architecture with skip connections to preserve input structure.
- **Discriminator:** Uses PatchGAN to judge local image patches of an input-output pair.
- **Loss:** Adversarial loss encourages realistic outputs; L1 loss encourages the output to match the target.
- **Dataset:** Facades paired images; each training example contains an input map and its corresponding facade photograph.
- **Limitation:** Few epochs may produce blurry or inaccurate results. This notebook demonstrates the method, not a production-quality image translator.
